# Where does a chord let go?

Four pure tones repeat three times a second. Pull their onsets apart in time and, somewhere,
the red one stops being part of the chord. This notebook is about finding that point by ear.

On every trial the red tone (2241 Hz) is louder on the last repetition in one of two sounds, and
you say which. The loudness of everything is shuffled so that only one comparison works:

| | what still works | easy if you hear |
|---|---|---|
| **A** | red against the other tones at that moment | one sound |
| **B** | red against its own earlier repetitions | the red tone on its own |

If shearing the chord unbinds the red tone, A gets harder and B gets easier. Where they trade
places is the bound.

| part | |
|---|---|
| 1 | the picture |
| 2 | two ways to hear the last tone |
| 3 | explorer |
| 4 | every step, by ear |
| 5 | the controls |
| 6 | the model |
| 7 | five imaginary listeners |
| 8 | measure your own bound |
| 9 | your results |
| 10 | build your own |
| 11 | for modellers |

**How to use it.** Headphones. Run the setup cell, then set your volume once on the level check
and leave it alone: every sound here plays at its real level, as in the lab. Move around with
the outline (☰, left). The code is hidden; double-click a title to see it.

In [ ]:
#@title setup (run this first)
import sys, subprocess, importlib.util, io, base64, wave, math, csv, json, time
from pathlib import Path
from collections import defaultdict
from functools import lru_cache
REF = 'adbf82623c7e34dbdf9e5e41946ac8b973fd259c'
REPO = 'https://github.com/MeysamAmirsardari/SeqSFG_task.git'
try:
    import tcbound.config
except Exception:
    if bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab')):
        ROOT = Path('/content') / ('seqsfg-' + REF[:12])
        if not ROOT.exists():
            subprocess.run(['git', 'clone', '-q', '--no-checkout', REPO, str(ROOT)], check=True)
            subprocess.run(['git', '-C', str(ROOT), 'checkout', '-q', '--detach', REF], check=True)
    else:
        ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'tcbound/config.py').exists()), None)
    for _m in [k for k in list(sys.modules) if k.split('.')[0] in ('tcbound', 'tshear', 'tcoh')]:
        del sys.modules[_m]
    sys.path.insert(0, str(ROOT))

get_ipython().run_line_magic('matplotlib', 'inline')
import numpy as np, matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, PathPatch
from matplotlib.path import Path as MPath
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display
import ipywidgets as W

from tcbound.config import Config, Condition, ConfigError, conditions, delta_db, srs_db, validate
from tcbound.stimulus import build_trial, render_interval
from tcbound.observer import CUE_LABELS, SimulatedListener, ceilings, cue_functions, cues, draw_gains
from tcbound.model import (curve, envelopes, half_way_ms, equivalent_shear_ms, target_coherence,
                           MODEL_FS, VARIANTS)
from tcbound.track import Track
from tcbound import analysis as AN
from tcbound.runner import Runner
from tcoh.model import coherence_matrix

CFG = Config(sample_rate=24000)        # 24 kHz is plenty for tones up to 4 kHz, and half the size
SR, T, OTH = CFG.sample_rate, CFG.target_index, CFG.others
STEPS = list(CFG.steps_ms)
FREQS = CFG.freqs_hz

INK, MUTED, FAINT, PAPER = '#1f2933', '#7b8794', '#e4e7eb', '#f7f8fa'
RED, OTHER = '#e03e52', '#52606d'
CA, CB, MODEL = '#2c6fbb', '#e07b26', '#9aa5b1'
TASK_COL = {'A': CA, 'B': CB, 'none': INK}
plt.rcParams.update({'figure.dpi': 120, 'font.size': 8, 'axes.edgecolor': '#9aa5b1',
                     'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 9, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
                     'axes.titlecolor': INK, 'legend.frameon': False})
LEV = (-14.0, 26.0)
CM_T = LinearSegmentedColormap.from_list('t', ['#fde6e9', '#f4a3ad', RED, '#8c1225'])
CM_O = LinearSegmentedColormap.from_list('o', ['#eef1f5', '#a9b8c8', '#52606d', '#1f2933'])
MIN_ANSWER_S = 0.4


# ---- trials -------------------------------------------------------------------------------
def cond(kind, step=0.0):
    # none | A | B | A_cap | B_cap | alone | flat_alone
    step = float(step)
    if kind == 'none':
        return Condition('none', 'none', step, 'figure')
    if kind in ('A', 'B'):
        return Condition(f'{kind}_{step:g}', kind, step, 'figure')
    if kind in ('A_cap', 'B_cap'):
        return Condition(kind, kind[0], step, 'capture')
    if kind == 'alone':
        return Condition('B_alone', 'B', step, 'alone')
    if kind == 'flat_alone':
        return Condition('flat_alone', 'none', step, 'alone')
    raise ValueError(kind)

def trial(kind, step=0.0, db=6.0, seed=None, cfg=CFG, target_position=1):
    rng = np.random.default_rng(seed)
    c = cond(kind, step)
    if db <= 0:
        return build_trial(cfg, c, cfg.srs_min_db, rng, target_position=target_position,
                           delta_override_db=0.0)
    return build_trial(cfg, c, srs_db(min(db, cfg.delta_max_db)), rng,
                       target_position=target_position)

def sounds(tr, cfg=CFG):
    return [render_interval(cfg, iv, tr.phases, tr.captor_phases, tr.total_ms)
            for iv in tr.intervals()]

def trial_wave(tr, cfg=CFG):
    a, b = sounds(tr, cfg)
    return np.concatenate([a, np.zeros(int(round(cfg.isi_ms * cfg.sample_rate / 1000))), b])


# ---- audio: every sound at its real level, as in the lab ---------------------------------------
def _b64(x, sr=SR):
    d = (np.clip(np.asarray(x, float), -1, 1) * 32767).astype('<i2')
    b = io.BytesIO()
    with wave.open(b, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes(d.tobytes())
    return base64.b64encode(b.getvalue()).decode()

def audio_tag(x, autoplay=False, controls=True, width=250):
    return (f'<audio {"controls" if controls else ""} {"autoplay" if autoplay else ""} '
            f'preload="{"auto" if autoplay else "none"}" style="height:32px;width:{width}px" '
            f'src="data:audio/wav;base64,{_b64(x)}"></audio>')

def pill(text, color):
    return (f'<span style="background:{color}18;color:{color};padding:2px 9px;border-radius:11px;'
            f'font-weight:600;font-size:12px;white-space:nowrap">{text}</span>')

def pairs(rows, heads=('louder at the end', 'not louder'), note=''):
    # rows: (label, colour, sound 1, sound 2). One row, one trial: the two sounds side by side.
    th = 'style="text-align:left;padding:4px 12px;color:#7b8794;font-weight:500"'
    h = ['<table style="border-collapse:collapse;font:13px system-ui,sans-serif">',
         f'<tr><th {th}></th><th {th}>{heads[0]}</th><th {th}>{heads[1]}</th></tr>']
    for lab, col, a, b in rows:
        h.append('<tr style="border-top:1px solid #edf0f3">'
                 f'<td style="padding:6px 12px">{pill(lab, col) if lab else ""}</td>'
                 f'<td style="padding:6px 12px">{audio_tag(a)}</td>'
                 f'<td style="padding:6px 12px">{audio_tag(b)}</td></tr>')
    h.append('</table>')
    if note:
        h.append(f'<div style="font:12px system-ui;color:#7b8794;margin:6px 12px">{note}</div>')
    display(HTML(''.join(h)))

def trial_row(label, kind, step, db, cfg=CFG):
    tr = trial(kind, step, db, cfg=cfg)
    a, b = sounds(tr, cfg)
    return (label, TASK_COL.get(kind[0] if kind[0] in 'AB' else 'none', INK), a, b)


# ---- pictures --------------------------------------------------------------------------------
def _lev(cm, g):
    return cm(float(np.clip((g - LEV[0]) / (LEV[1] - LEV[0]), 0, 1)))

def roll(ax, iv, cfg=CFG, flat=False, numbers=True, xmax=None, mark=None):
    # one interval: a bar per tone per repetition, darker when louder, numbers in dB
    o = iv.onsets_ms - cfg.lead_ms
    R, K = o.shape
    x1 = np.nanmax(o) + cfg.tone_ms
    ax.axvspan(np.nanmin(o[-1]) - 14, x1 + 14, color='#eef1f5', zorder=0, lw=0)
    ax.text(x1 + 14, K - .36, mark or 'last', ha='right', va='center', fontsize=6.5,
            color=RED if mark else MUTED)
    if mark:
        ax.add_patch(Rectangle((o[-1, cfg.target_index] - 7, cfg.target_index - .45), cfg.tone_ms + 14,
                               .9, fc='none', ec=RED, lw=1.3, zorder=5))
    if iv.captor_onsets_ms is not None:
        co = iv.captor_onsets_ms - cfg.lead_ms
        for r in range(co.shape[0]):
            for k in range(K):
                if np.isfinite(co[r, k]):
                    fc = OTHER if flat else _lev(CM_O, iv.captor_gains_db[r, k])
                    ax.add_patch(Rectangle((co[r, k], k - .3), cfg.tone_ms, .6, fc=fc, alpha=.45,
                                           ec=OTHER, lw=.6, ls=(0, (2, 2)), zorder=2))
    for r in range(R):
        for k in range(K):
            if not np.isfinite(o[r, k]):
                continue
            g = iv.gains_db[r, k]
            tgt = k == cfg.target_index
            fc = (RED if tgt else OTHER) if flat else _lev(CM_T if tgt else CM_O, g)
            ax.add_patch(Rectangle((o[r, k], k - .33), cfg.tone_ms, .66, fc=fc, ec='none', zorder=3))
            if numbers and not flat and (tgt or r == R - 1):
                ax.text(o[r, k] + cfg.tone_ms / 2, k, f'{g:+.0f}', ha='center', va='center',
                        fontsize=6.2, color='white' if g > 4 else INK, zorder=4)
    ax.set_xlim(-25, (xmax if xmax is not None else x1) + 25)
    ax.set_ylim(-.7, K - .1)
    ax.set_yticks(range(K)); ax.set_yticklabels([f'{f:.0f}' for f in cfg.freqs_hz])
    ax.tick_params(axis='y', length=0)
    for lab in ax.get_yticklabels():
        if lab.get_text() == f'{cfg.target_freq_hz:.0f}':
            lab.set_color(RED); lab.set_fontweight('bold')
    ax.set_xlabel('ms', color=MUTED)
    ax.spines['left'].set_visible(False)

@lru_cache(maxsize=256)
def coh(kind, step, variant='halfwave'):
    c = cond('A' if kind in ('none', 'A', 'B') else kind, step)
    if c.kind == 'alone':
        return None
    M = np.asarray(coherence_matrix(envelopes(CFG, c), MODEL_FS, **VARIANTS[variant]).matrix, float)
    d = np.sqrt(np.clip(np.diag(M), 1e-300, None))
    return M / np.outer(d, d)

def web(ax, R, cfg=CFG, title=True):
    # the model's coherence between the tones: thicker arcs, stronger glue
    K, t = len(cfg.freqs_hz), cfg.target_index
    if R is None:
        ax.text(.5, .5, 'no other tones', ha='center', va='center', color=MUTED,
                transform=ax.transAxes); ax.axis('off'); return
    for i in range(K):
        for j in range(i + 1, K):
            c = float(np.clip(R[i, j], 0, 1))
            bend = .25 + .32 * (j - i)
            p = MPath([(0, i), (bend, (i + j) / 2), (0, j)], [MPath.MOVETO, MPath.CURVE3, MPath.CURVE3])
            red = t in (i, j)
            ax.add_patch(PathPatch(p, fc='none', ec=RED if red else OTHER, lw=.3 + 6.5 * c ** 1.6,
                                   alpha=(.25 + .65 * c) if red else (.12 + .4 * c), capstyle='round'))
    for k in range(K):
        ax.plot(0, k, 'o', ms=10, color=RED if k == t else OTHER, mec='white', mew=1.5, zorder=3)
    ax.set_xlim(-.25, 1.35); ax.set_ylim(-.7, K - .3); ax.axis('off')
    if title:
        rho = float(np.mean([R[t, k] for k in range(K) if k != t]))
        ax.set_title(f'model glue  {rho:.2f}', color=RED if rho > .5 else MUTED, fontsize=8.5)

def pair_figure(tr, cfg=CFG, heads=('louder at the end', 'not louder'), web_R='auto', title=None):
    widths = [5, 5, 1.6] if web_R is not None else [5, 5]
    fig = plt.figure(figsize=(10.6, 2.35))
    gs = fig.add_gridspec(1, len(widths), width_ratios=widths, wspace=.08)
    xmax = max(np.nanmax(iv.onsets_ms) for iv in tr.intervals()) - cfg.lead_ms + cfg.tone_ms
    sig_first = tr.target_position == 1
    for j, iv in enumerate(tr.intervals()):
        ax = fig.add_subplot(gs[j])
        roll(ax, iv, cfg, xmax=xmax)
        ax.set_title(heads[0] if (j == 0) == sig_first else heads[1], color=MUTED,
                     fontweight='normal', fontsize=8.5)
        if j:
            ax.set_yticklabels([])
    if web_R is not None:
        R = coh(_kind_of(tr), tr.step_ms) if isinstance(web_R, str) else web_R
        web(fig.add_subplot(gs[-1]), R)
    if title:
        fig.suptitle(title, x=.125, ha='left', y=1.04, fontsize=10, fontweight='bold', color=INK)
    plt.show()

def _kind_of(tr):
    if tr.kind == 'alone':
        return 'alone'
    if tr.kind == 'capture':
        return f'{tr.task}_cap'
    return tr.task


# ---- numbers ---------------------------------------------------------------------------------
CEIL = {k: ceilings(CFG, cond(k), n=8000) for k in ('A', 'B', 'alone')}

def shortcut_pc(kind, db, n=3000, seed=0):
    # % correct of an ideal listener who has only the best scrambled cue
    c = cond(kind)
    ce = ceilings(CFG, c, n=8000)
    if not ce['lowest_shortcut'] or db <= 0:
        return None, 50.0
    f = cue_functions(CFG)[ce['lowest_shortcut']]
    rng = np.random.default_rng(seed)
    d = f(draw_gains(CFG, c, db, True, n, rng)) - f(draw_gains(CFG, c, db, False, n, rng))
    return CUE_LABELS[ce['lowest_shortcut']], 100 * (np.mean(d > 0) + .5 * np.mean(d == 0))

RHO_GRID = np.linspace(0, 72, 145)
RHO = {v: curve(CFG, RHO_GRID, v) for v in ('halfwave', 'magnitude')}
HALF = {v: half_way_ms(CFG, v) for v in ('halfwave', 'magnitude')}
RHO_CAP = target_coherence(CFG, cond('A_cap'))

display(HTML(
    '<div style="font:13px system-ui;padding:12px 16px;border-radius:10px;'
    'background:linear-gradient(90deg,#0b1021,#1d2a4d 55%,#4a1d2e);color:#e8ecf3;max-width:760px">'
    '<b style="font-size:15px">ready</b>&nbsp;&nbsp; four tones '
    + ' · '.join(f'{f:.0f}' for f in FREQS) + ' Hz, the red one at '
    f'{CFG.target_freq_hz:.0f} Hz &nbsp;·&nbsp; {CFG.tone_ms:g} ms tones at {CFG.rate_hz:g} Hz, '
    f'{CFG.n_repeats} repetitions &nbsp;·&nbsp; shear {", ".join(f"{s:g}" for s in STEPS)} ms'
    f' &nbsp;·&nbsp; rove ±{CFG.rove_db:g} dB</div>'))

In [ ]:
#@title level check
tr_t, tr_c = trial('flat_alone', 0, 0, seed=1), trial('none', 0, 0, seed=1)
pairs([('', INK, sounds(tr_t)[1], sounds(tr_c)[1])], heads=('the red tone alone', 'all four together'),
      note='Set a comfortable level on these, then leave the volume alone for the rest of the '
           'notebook. Tones will range 12 dB either side of this.')

---
# 1 · The picture

In [ ]:
#@title the chord as light
from scipy.ndimage import gaussian_filter
t_ms = np.linspace(-30, 300, 1000)
s_ms = np.linspace(0, 72, 440)
COLS = np.array([[0.22, 0.45, 1.00],     # 683 Hz
                 [0.05, 0.92, 0.70],     # 1229 Hz
                 [1.00, 0.16, 0.30],     # 2241 Hz, the red tone
                 [1.00, 0.78, 0.14]])    # 3985 Hz
img = np.zeros((s_ms.size, t_ms.size, 3))
for k in range(4):
    x = t_ms[None, :] - k * s_ms[:, None]
    g = np.clip(np.minimum(x, CFG.tone_ms - x) / CFG.ramp_ms, 0, 1)
    img += (np.sin(g * np.pi / 2) ** 2)[:, :, None] * COLS[k]
light = 1 - np.exp(-(1.25 * img + 1.1 * gaussian_filter(img, sigma=(4, 9, 0))))

BG = '#05070d'
fig = plt.figure(figsize=(11, 6.4), facecolor=BG)
gs = fig.add_gridspec(1, 2, width_ratios=[3.1, 1], wspace=.06)
ax = fig.add_subplot(gs[0], facecolor=BG)
ax.imshow(light, extent=[t_ms[0], t_ms[-1], s_ms[-1], s_ms[0]], aspect='auto', interpolation='bilinear')
for s in STEPS:
    ax.axhline(s, color='white', lw=.5, alpha=.22, ls=(0, (1, 3)))
    ax.text(-34, s, f'{s:g} ms', color='#c9d1de', fontsize=8, ha='right', va='center')
ax.set_xlim(t_ms[0], t_ms[-1]); ax.set_ylim(s_ms[-1], s_ms[0])
ax.set_yticks([]); ax.set_xticks([0, 100, 200, 300])
ax.tick_params(colors='#7f8ba3'); ax.set_xlabel('time within one repetition (ms)', color='#9aa5b8')
for sp in ax.spines.values():
    sp.set_visible(False)
for k, f in enumerate(FREQS):
    ax.text(k * s_ms[-1] + 30, s_ms[-1] - 1.6, f'{f:.0f} Hz', color=BG, fontsize=8,
            ha='center', va='bottom', fontweight='bold')
for v, ls in (('halfwave', '-'), ('magnitude', (0, (4, 3)))):
    ax.axhline(HALF[v], color='#ff5a6e', lw=.9, ls=ls, alpha=.55)

ax2 = fig.add_subplot(gs[1], facecolor=BG, sharey=ax)
for v, ls, lab in (('halfwave', '-', 'half-wave'), ('magnitude', '--', 'magnitude')):
    ax2.plot(RHO[v], RHO_GRID, ls, color='#f2f4f8', lw=1.8, alpha=.95 if v == 'halfwave' else .6)
    hw = HALF[v]
    rh = float(np.interp(hw, RHO_GRID, RHO[v]))
    ax2.axhline(hw, color='#ff5a6e', lw=.9, ls='-' if v == 'halfwave' else (0, (4, 3)), alpha=.55)
    ax2.plot(rh, hw, 'o', ms=7, color=RED, mec=BG, mew=1.5, zorder=4)
    ax2.text(rh, hw - 2.2, f'{lab}\nlets go at {hw:.0f} ms', color='#f2f4f8', fontsize=8,
             ha='center', va='bottom', alpha=.95 if v == 'halfwave' else .7, linespacing=1.3)
ax2.set_xlim(0.05, 1.04); ax2.invert_xaxis()
ax2.set_xticks([1, .5, .1]); ax2.set_xticklabels(['bound', '', 'free'])
ax2.tick_params(colors='#7f8ba3', left=False, labelleft=False)
ax2.set_xlabel('how bound the red tone is (model)', color='#9aa5b8')
for sp in ax2.spines.values():
    sp.set_visible(False)
fig.text(.125, .965, 'Where does a chord let go?', color='white', fontsize=17, fontweight='bold')
fig.text(.125, .925, 'Four tones, one colour each. Where they overlap the light adds up to white: '
         'one sound. Shear them, and the colours come apart.', color='#aeb8c9', fontsize=9.5)
plt.show()

Downwards the shear grows: each tone starts later than the one below it by that many ms. The
right panel is the model's answer to how bound the red tone is. Its two readings disagree about
where the red tone lets go, 18 or 35 ms, and the experiment is built to tell which is right.

In [ ]:
#@title one sound, four sounds
pairs([('', INK, sounds(trial('none', 0, 0, seed=3))[1], sounds(trial('none', 64, 0, seed=3))[1])],
      heads=('shear 0 ms', 'shear 64 ms'))

---
# 2 · Two ways to hear the last tone

Each bar is one tone in one repetition, darker when louder; the numbers are levels in dB. In
both rows the left sound is the one with the louder red tone, and the draw was picked so that
the comparison the type scrambles points the wrong way.

In [ ]:
#@title the two trial types
def _find(kind, misleads, step=0.0, db=6.0):
    for seed in range(4000):
        tr = trial(kind, step, db, seed=seed)
        if misleads(tr):
            return tr

def _own(g):
    return g[-1, T] - np.mean(g[:-1, T])
def _tim(g):
    return g[-1, T] - np.mean(g[-1, OTH])

trA = _find('A', lambda tr: _own(tr.second.gains_db) > _own(tr.first.gains_db) + 3)
trB = _find('B', lambda tr: _tim(tr.second.gains_db) > _tim(tr.first.gains_db) + 3, step=64.0)
pair_figure(trA, title='A   the last repetition is reset to one level for all four')
display(HTML(f'<div style="font:12.5px system-ui;margin:-6px 0 14px 70px;color:{CA}">'
             'works: red against the others in the last column &nbsp;·&nbsp; '
             '<span style="color:#7b8794">misleads here: red against its own past</span></div>'))
pair_figure(trB, title='B   the other three jump, the red tone holds')
display(HTML(f'<div style="font:12.5px system-ui;margin:-6px 0 14px 70px;color:{CB}">'
             'works: red against its own earlier repetitions &nbsp;·&nbsp; '
             '<span style="color:#7b8794">misleads here: red against the others</span></div>'))
pairs([('A · 0 ms', CA, *sounds(trA)), ('B · 64 ms', CB, *sounds(trB))])

The first four repetitions are made by the same code in A and B, so nothing tells you the type
until the last one. Even the jumps on the last repetition look alike: the other three move by
the same random amount in both, and so does the red tone relative to them. What differs is only
which comparison stays exact.

---
# 3 · Explorer

Move the shear, pick a trial type, listen. The left sound is always the louder one. On the
right, the model's glue between the tones: the red arcs are the ones that matter.

In [ ]:
#@title explorer
_wide = {'description_width': 'initial'}
x_step = W.FloatSlider(value=16, min=0, max=72, step=1, description='shear (ms)',
                       continuous_update=False, style=_wide, layout=W.Layout(width='380px'))
x_kind = W.ToggleButtons(options=[('no rove', 'none'), ('A', 'A'), ('B', 'B'),
                                  ('A + captors', 'A_cap'), ('B + captors', 'B_cap'),
                                  ('red alone', 'alone')], value='A',
                         style={'button_width': '92px'})
x_db = W.FloatSlider(value=6, min=0, max=12, step=.5, description='louder by (dB)',
                     continuous_update=False, style=_wide, layout=W.Layout(width='380px'))
x_new = W.Button(description='new draw', icon='refresh')
x_out = W.Output()
_X = {'seed': 0}

def _explore(*_):
    kind, step, db = x_kind.value, float(x_step.value), float(x_db.value)
    tr = trial(kind, step, db, seed=_X['seed'])
    with x_out:
        x_out.clear_output(wait=True)
        pair_figure(tr)
        pairs([('', TASK_COL.get(kind[0], INK), *sounds(tr))])
        cue, pc = shortcut_pc(kind if kind != 'none' else 'A', db) if kind != 'none' else (None, None)
        bits = []
        if kind not in ('none', 'alone'):
            r = coh(kind, step)
            bits.append(f'model glue {np.mean([r[T, k] for k in OTH]):.2f}')
        if cue:
            bits.append(f'a listener with only the best shortcut ({cue}) would score {pc:.0f}% '
                        f'at {db:g} dB')
        if kind == 'none':
            bits.append('no rove: every comparison works, so this says nothing about binding')
        display(HTML(f'<div style="font:12px system-ui;color:#52606d;margin:4px 12px">'
                     + ' &nbsp;·&nbsp; '.join(bits) + '</div>'))

def _redraw(_=None):
    _X['seed'] += 1
    _explore()

for w_ in (x_step, x_kind, x_db):
    w_.observe(_explore, names='value')
x_new.on_click(_redraw)
display(W.VBox([x_kind, W.HBox([x_step, x_db, x_new]), x_out]))
_explore()

---
# 4 · Every step, by ear

One section per shear. The picture shows where the tones sit and how much glue the model sees
between them. Then five trials, left always louder: no rove, then A and B at two sizes. If the
idea is right, A gets harder and B easier as you go down the page. Run a cell again for new draws.

In [ ]:
#@title helpers for this part
ROWS = (('no rove · 4 dB', 'none', 4.0), ('A · 6 dB', 'A', 6.0), ('A · 10 dB', 'A', 10.0),
        ('B · 6 dB', 'B', 6.0), ('B · 10 dB', 'B', 10.0))

def step_section(step):
    fig = plt.figure(figsize=(10.6, 2.0))
    gs = fig.add_gridspec(1, 2, width_ratios=[8.4, 1.6], wspace=.05)
    ax = fig.add_subplot(gs[0])
    roll(ax, trial('none', step, 0).standard, flat=True, mark='louder here, in one of the two sounds')
    web(fig.add_subplot(gs[1]), coh('A', step))
    plt.show()
    pairs([trial_row(lab, kind, step, db) for lab, kind, db in ROWS])
print('ready')

## 0 ms
All four start together. The model calls the red tone fully bound.

In [ ]:
#@title picture and sounds
step_section(0)

## 8 ms
Each tone starts 8 ms after the one below it, so red starts 16 ms after the lowest.

In [ ]:
#@title picture and sounds
step_section(8)

## 16 ms
The half-wave reading of the model puts the bound near here.

In [ ]:
#@title picture and sounds
step_section(16)

## 32 ms
Adjacent tones still overlap by 28 ms. The magnitude reading still sees a lot of glue.

In [ ]:
#@title picture and sounds
step_section(32)

## 64 ms
No two tones overlap any more. Both readings say the red tone is mostly free.

In [ ]:
#@title picture and sounds
step_section(64)

---
# 5 · The controls

## Captors

The other three tones also play on their own, half-way between repetitions. Every onset is
still synchronous, but the others now repeat at 6 Hz and the red tone at 3 Hz, so their rhythms
no longer match. The model's glue drops to 0.71, the value a 13 ms shear gives. If binding is
coherence, this condition lands where 13 ms lands, without anything being fitted to it.

In [ ]:
#@title picture and sounds
pair_figure(trial('A_cap', 0, 6, seed=2))
pairs([trial_row('A + captors · 6 dB', 'A_cap', 0, 6), trial_row('A + captors · 10 dB', 'A_cap', 0, 10),
       trial_row('B + captors · 6 dB', 'B_cap', 0, 6), trial_row('B + captors · 10 dB', 'B_cap', 0, 10)],
      note='Pale dashed bars are the captors.')

## The red tone alone

Type B with nothing else playing: the best the own-stream comparison can do. B at the largest
shear should come close to this if the red tone is truly on its own there.

In [ ]:
#@title picture and sounds
pair_figure(trial('alone', 0, 3, seed=4), web_R=None)
pairs([trial_row(f'red alone · {d:g} dB', 'alone', 0, d) for d in (1.5, 3.0, 6.0)])

## No shortcuts

How big the increase must be before a listener who uses only one cue reaches 79% correct,
computed for an ideal listener on the real draws. Stars are the cue each type keeps, which needs
no increase at all. Every other cue needs at least 8.5 dB, so a threshold of 3 or 4 dB means the
kept cue was used.

In [ ]:
#@title the shortcuts
cues_ = list(CUE_LABELS)
fig, ax = plt.subplots(figsize=(9.2, 3.5))
ax.axvspan(0, 8.5, color=PAPER, lw=0, zorder=0)
for i, c in enumerate(cues_):
    for off, kind, col in ((-.17, 'A', CA), (.17, 'B', CB)):
        v = CEIL[kind]['per_cue_db'][c]
        y = i + off
        if v == 0:
            ax.plot(0.25, y, marker='*', ms=13, color=col, zorder=3)
            ax.text(0.8, y, f'kept by {kind}', color=col, va='center', fontsize=7.5, fontweight='bold')
        else:
            v = min(v, 15.5)
            ax.plot([0, v], [y, y], color=col, lw=2.2, alpha=.35, solid_capstyle='round')
            ax.plot(v, y, 'o', color=col, ms=6.5, zorder=3)
ax.axvline(8.5, color=RED, lw=1, ls=':')
ax.text(8.6, -.75, ' 8.5 dB: the easiest shortcut', color=RED, fontsize=7.5, va='bottom')
ax.set_yticks(range(len(cues_))); ax.set_yticklabels([CUE_LABELS[c] for c in cues_])
ax.invert_yaxis(); ax.set_xlim(0, 16); ax.tick_params(axis='y', length=0)
ax.set_xlabel('increase needed for 79% correct using that cue alone (dB)')
ax.plot([], [], 'o', color=CA, label='type A'); ax.plot([], [], 'o', color=CB, label='type B')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

---
# 6 · The model

The coherence between every pair of tones, from the temporal-coherence model (rate filters 2 to
32 Hz, as validated in `tcoh`). Dark is coherent. The red row is the red tone's glue to the rest.

In [ ]:
#@title coherence at each step
panels = [(f'{s:g} ms', coh('A', s)) for s in STEPS] + [('captors', coh('A_cap', 0))]
cm = LinearSegmentedColormap.from_list('c', ['#ffffff', '#c9d6e3', '#52606d', '#111a24'])
fig, axs = plt.subplots(1, len(panels), figsize=(11.4, 2.35))
for ax, (lab, R) in zip(axs, panels):
    ax.imshow(np.clip(R, 0, 1), cmap=cm, vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{R[i, j]:.2f}', ha='center', va='center', fontsize=6.3,
                    color='white' if R[i, j] > .55 else INK)
    ax.add_patch(Rectangle((-.5, T - .5), 4, 1, fc='none', ec=RED, lw=1.6))
    rho = np.mean([R[T, k] for k in OTH])
    ax.set_title(f'{lab}\nglue {rho:.2f}', fontsize=8.5, color=RED if lab == 'captors' else INK)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels([f'{f/1000:.1f}k' for f in FREQS], fontsize=6.5)
    ax.set_yticklabels([f'{f/1000:.1f}k' for f in FREQS], fontsize=6.5)
    for sp in ax.spines.values():
        sp.set_visible(False)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7.6, 3.2))
for v, ls in (('halfwave', '-'), ('magnitude', '--')):
    ax.plot(RHO_GRID, RHO[v], ls, color=INK, lw=1.8, label=f'{v} reading', alpha=1 if v == 'halfwave' else .6)
    ax.plot(STEPS, curve(CFG, STEPS, v), 'o', color=INK, ms=4.5, alpha=1 if v == 'halfwave' else .6)
    hw = HALF[v]
    ax.axvline(hw, color=RED, lw=1, ls=ls, alpha=.7)
    ax.text(hw + .8, 1.03, f'lets go at {hw:.0f} ms', color=RED, fontsize=7.5, alpha=1 if v == 'halfwave' else .7)
ax.axhline(RHO_CAP, color=MODEL, lw=1, ls=':')
eq = equivalent_shear_ms(CFG, RHO_CAP)
ax.plot(eq, RHO_CAP, 'D', color=MODEL, ms=5)
ax.text(eq + 1.2, RHO_CAP + .02, f'captors at 0 ms = a {eq:.0f} ms shear', color=MUTED, fontsize=7.5)
ax.set_ylim(0, 1.1); ax.set_xlim(-1, 73); ax.set_xticks(STEPS)
ax.set_xlabel('shear (ms)'); ax.set_ylabel('glue of the red tone')
ax.legend(loc='lower left', fontsize=8); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

The two readings differ in what the glue is made of. The half-wave reading lets the fast rate
filters dominate, and they lose the 60 ms pulses as soon as these stop overlapping. The magnitude
reading leans on the slow 3 Hz repetition, which a 64 ms shift barely moves. The steps were chosen
so that the two predict clearly different curves.

---
# 7 · Five imaginary listeners

The real runner and the real analysis, run on simulated listeners: twelve sessions for the two
whose binding changes, four for the rest. Left, the binding index from thresholds (B minus A, positive when bound). Right,
the timbre weight read from every single response (1 when only the timbre of the last chord is
used, 0 when only the red tone's own past). The dot on each curve is that listener's bound.

In [ ]:
#@title simulate them (about 30 s)
class Follows(SimulatedListener):
    variant = 'halfwave'
    def binding(self, c):
        if c.kind == 'alone':
            return 0.0
        if c.kind == 'capture':
            return target_coherence(self.cfg, c, self.variant)
        return float(curve(self.cfg, [c.step_ms], self.variant)[0])

def imaginary(make, n_sessions, key):
    rows, th = [], defaultdict(list)
    for s in range(n_sessions):
        r = Runner(CFG, None, auto='coherence', seed=1000 * key + s, dry=True)
        r.sim = make()
        r.start_dry(code=f'IMG{key}_{s}')
        r.main_block()
        rows += r.rows
        for t_ in r.tracks.values():
            if t_.threshold_db() is not None:
                th[t_.condition].append(t_.threshold_db())
    thr = {k: float(np.mean(v)) for k, v in th.items()}
    return AN.binding_curve(thr, CFG), AN.weight_curve(AN.decision_weights(rows, CFG), CFG)

def _follows(v):
    def make():
        s = Follows(CFG, 'coherence'); s.variant = v; return s
    return make

WHO = {'follows the model, half-wave': ('#e03e52', _follows('halfwave')),
       'follows the model, magnitude': ('#8e44ad', _follows('magnitude')),
       'shear changes nothing': ('#7b8794', lambda: SimulatedListener(CFG, 'null')),
       'always one sound': (CA, lambda: SimulatedListener(CFG, 'fused')),
       'always the red tone alone': (CB, lambda: SimulatedListener(CFG, 'within'))}
IMAGINARY = {name: imaginary(make, 12 if name.startswith('follows') else 4, key)
             for key, (name, (col, make)) in enumerate(WHO.items())}

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for name, (col, _) in WHO.items():
    bc, wc = IMAGINARY[name]
    axs[0].plot(STEPS, bc['bi'], '-o', color=col, lw=2, ms=4, label=name)
    axs[1].plot(STEPS, wc['w'], '-o', color=col, lw=2, ms=4)
    if wc['s50_ms'] is not None:
        y = float(np.interp(wc['s50_ms'], STEPS, wc['w']))
        axs[1].plot(wc['s50_ms'], y, 'o', ms=11, mfc='white', mec=col, mew=2, zorder=4)
        axs[1].text(wc['s50_ms'] + 1.5, y + .04, f"{wc['s50_ms']:.0f} ms", color=col, fontsize=7.5)
axs[0].axhline(0, color=MUTED, lw=.6)
axs[0].set_ylabel('binding index (dB)'); axs[1].set_ylabel('timbre weight')
axs[1].set_ylim(-.12, 1.15)
for ax in axs:
    ax.set_xticks(STEPS); ax.set_xlabel('shear (ms)'); ax.grid(alpha=.2)
fig.legend(*axs[0].get_legend_handles_labels(), loc='upper center', ncol=5, fontsize=8,
           bbox_to_anchor=(.5, 1.07))
plt.tight_layout(); plt.show()

Only the two listeners whose binding really changes produce falling curves, and their bounds
land where the two readings of the model put them. These listeners went through the same
trials, staircases and analysis as you will. One person's bound is noisy, more so when the curve
is shallow, which is why the lab protocol pools two sessions per person and states the bound for
the group.

---
# 8 · Measure your own bound

The real staircases, interleaved as in the lab, with the real trials. Each trial plays once:
sound 1, a short pause, sound 2. Answer 1 or 2. Pick a session length, press start, and keep your
eyes off the code. You can stop whenever you like and press continue later; nothing is lost until
the runtime restarts.

| session | conditions | time |
|---|---|---|
| quick | A and B at 0, 16 and 64 ms | about 15 to 20 min |
| screen | A and B at 0 and 64 ms | about 10 min |
| full | all 13, as in the lab | about 35 to 40 min |

In [ ]:
#@title the experiment
PRESETS = {'quick': dict(steps_ms=(0.0, 16.0, 64.0), include_capture=False, include_alone=False),
           'screen': dict(steps_ms=(0.0, 64.0), include_capture=False, include_alone=False),
           'full': dict()}
EXP = globals().get('EXP')

class Session:
    def __init__(self, preset, feedback):
        self.preset, self.feedback = preset, feedback
        self.cfg = CFG.replace(**PRESETS[preset])
        self.conds = {c.name: c for c in conditions(self.cfg)}
        self.tracks = {n: Track(self.cfg, n) for n in self.conds}
        self.rng = np.random.default_rng()
        self.queue, self.log, self.prev = [], [], None
        self.cur, self.shown, self.plays, self.dur = None, 0.0, 0, 0.0
        self.est = int(round(len(self.tracks) * 27 * (1 + self.cfg.catch_rate)))
        self.started = time.strftime('%Y-%m-%d %H:%M')

    def active(self):
        return [n for n, t in self.tracks.items() if not t.finished]

    def pick(self):
        act = self.active()
        if not act:
            return None
        if self.rng.random() < self.cfg.catch_rate:
            return 'catch'
        self.queue = [n for n in self.queue if n in act]
        if not self.queue:
            q = list(act); self.rng.shuffle(q)
            if len(q) > 1 and q[0] == self.prev:
                q.append(q.pop(0))
            self.queue = q
        self.prev = self.queue.pop(0)
        return self.prev

e_preset = W.ToggleButtons(options=[('quick', 'quick'), ('screen', 'screen'), ('full', 'full')],
                           value='quick', style={'button_width': '90px'})
e_fb = W.Checkbox(value=True, description='show if I was right', indent=False)
e_start = W.Button(description='start a new session', button_style='primary', icon='play')
e_cont = W.Button(description='continue', icon='forward')
e_again = W.Button(description='play again', icon='repeat', layout=W.Layout(width='130px'))
e1 = W.Button(description='1', layout=W.Layout(width='130px', height='58px'))
e2 = W.Button(description='2', layout=W.Layout(width='130px', height='58px'))
for b_ in (e1, e2):
    b_.style.font_weight = 'bold'
e_stage, e_note = W.Output(), W.HTML()
e_bar = W.IntProgress(min=0, max=100, layout=W.Layout(width='420px'))

def _lights(x, dur1, isi, key):
    tot = 2 * dur1 + isi
    p1, p2 = 100 * dur1 / tot, 100 * (dur1 + isi) / tot
    on, off = 'background:#1f2933;color:#fff;transform:scale(1.1)', 'background:#e4e7eb;color:#9aa5b1;transform:scale(1)'
    css = (f'<style>@keyframes tcbA{key}{{0%,{p1:.2f}%{{{on}}}{p1 + .01:.2f}%,100%{{{off}}}}}'
           f'@keyframes tcbB{key}{{0%,{p2:.2f}%{{{off}}}{p2 + .01:.2f}%,99.5%{{{on}}}100%{{{off}}}}}'
           '.tcbl{width:58px;height:58px;border-radius:50%;display:inline-flex;align-items:center;'
           f'justify-content:center;font:700 22px system-ui;margin-right:16px;{off};transition:none}}</style>')
    return (css + '<div style="display:flex;align-items:center;padding:6px 0 4px 2px">'
            + audio_tag(x, autoplay=True, controls=False, width=1)
            + f'<div class="tcbl" style="animation:tcbA{key} {tot:.3f}s linear 1">1</div>'
            + f'<div class="tcbl" style="animation:tcbB{key} {tot:.3f}s linear 1">2</div></div>')

def _play(_=None):
    S = EXP
    if S is None or S.cur is None:
        return
    name, c, tr = S.cur
    S.plays += 1
    with e_stage:
        e_stage.clear_output(wait=True)
        display(HTML(_lights(S.wave, tr.total_ms / 1000, S.cfg.isi_ms / 1000,
                             f'{len(S.log)}_{S.plays}')))
    S.shown = time.time()

def _status():
    S = EXP
    done = sum(t.finished for t in S.tracks.values())
    n = len(S.log)
    e_bar.max = max(S.est, n + 5 * len(S.active()) + 1)
    e_bar.value = n
    e_note.value = (f'<span style="font:12.5px system-ui;color:#52606d">trial {n + 1} &nbsp;·&nbsp; '
                    f'{done} of {len(S.tracks)} staircases finished</span>')

def _next():
    S = EXP
    name = S.pick()
    if name is None:
        S.cur = None
        e1.disabled = e2.disabled = e_again.disabled = True
        with e_stage:
            e_stage.clear_output()
        e_note.value = ('<b style="font:14px system-ui;color:#1e8449">All staircases finished.</b> '
                        '<span style="font:13px system-ui">Run the next part for your results.</span>')
        e_bar.value = e_bar.max
        return
    if name == 'catch':
        c = Condition('catch', 'none', 0.0, 'figure', 'catch')
        srs = srs_db(S.cfg.catch_delta_db)
    else:
        c, srs = S.conds[name], S.tracks[name].srs
    tr = build_trial(S.cfg, c, srs, S.rng, is_catch=name == 'catch')
    S.cur, S.wave, S.plays = (name, c, tr), trial_wave(tr, S.cfg), 0
    S.dur = (2 * tr.total_ms + S.cfg.isi_ms) / 1000
    e1.disabled = e2.disabled = e_again.disabled = False
    _status()
    _play()

def _answer(k):
    S = EXP
    if S is None or S.cur is None:
        return
    since = time.time() - S.shown
    if since < max(MIN_ANSWER_S, S.dur - 0.25):
        e_note.value = '<span style="font:12.5px system-ui;color:#b9770e">wait for sound 2 to finish</span>'
        return
    name, c, tr = S.cur
    ok = int(k == tr.target_position)
    if name != 'catch':
        S.tracks[name].update(bool(ok))
    c1, c2 = cues(S.cfg, tr.first.gains_db), cues(S.cfg, tr.second.gains_db)
    num = lambda v: float(v) if np.isfinite(v) else float('nan')
    S.log.append(dict(trial=len(S.log) + 1, phase='catch' if name == 'catch' else 'main',
                      condition=name, task=c.task, kind=c.kind, step_ms=c.step_ms,
                      srs_db=round(tr.srs_db, 3), delta_db=round(tr.delta_db, 3),
                      target_position=tr.target_position, response=str(k), correct=ok,
                      plays=S.plays, rt_s=round(since - S.dur, 3),
                      timbre_1=num(c1['timbre']), timbre_2=num(c2['timbre']),
                      own_1=num(c1['own']), own_2=num(c2['own']),
                      gains_1=json.dumps(np.round(tr.first.gains_db, 3).tolist()).replace('NaN', 'null'),
                      gains_2=json.dumps(np.round(tr.second.gains_db, 3).tolist()).replace('NaN', 'null'),
                      time=time.strftime('%H:%M:%S')))
    S.cur = None
    e1.disabled = e2.disabled = True
    if S.feedback:
        with e_stage:
            e_stage.clear_output(wait=True)
            msg = (('<div style="font:600 18px system-ui;color:#1e8449;padding:18px 4px">right</div>')
                   if ok else (f'<div style="font:600 18px system-ui;color:{RED};padding:18px 4px">'
                               f'it was {tr.target_position}</div>'))
            display(HTML(msg))
        time.sleep(0.55)
    _next()

def _start(_=None):
    global EXP
    EXP = Session(e_preset.value, e_fb.value)
    _next()

def _continue(_=None):
    if EXP is None:
        return _start()
    EXP.feedback = e_fb.value
    if EXP.cur is None and EXP.active():
        _next()
    elif EXP.cur is not None:
        _status(); _play()

e_start.on_click(_start); e_cont.on_click(_continue); e_again.on_click(_play)
e1.on_click(lambda _: _answer(1)); e2.on_click(lambda _: _answer(2))
e1.disabled = e2.disabled = e_again.disabled = True
display(W.VBox([W.HBox([e_preset, e_fb]), W.HBox([e_start, e_cont]),
                W.HTML('<div style="height:6px"></div>'), e_stage, W.HBox([e1, e2, e_again]),
                e_bar, e_note]))
if EXP is not None:
    _status()
    e_note.value += ' &nbsp;<span style="font:12.5px system-ui;color:#7b8794">(a session is open: press continue)</span>'

The play again button is there in case your browser blocks the first play. In the lab each
trial plays once, so the number of plays is recorded with every answer.

---
# 9 · Your results

Your staircases, your thresholds, and your timbre weights, analysed exactly as the lab data will
be. With one quick session expect noisy numbers: the lab protocol uses two full sessions per
person. Faint lines are the imaginary listeners from part 7, if you ran it.

In [ ]:
#@title results
S = globals().get('EXP')
if S is None or not S.log:
    print('no trials yet: run the experiment above first')
else:
    cfgx, names = S.cfg, list(S.tracks)
    stepsx = sorted(cfgx.steps_ms)

    # staircases
    ncol = min(5, len(names)); nrow = int(math.ceil(len(names) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(2.25 * ncol, 1.75 * nrow), sharey=True, squeeze=False)
    for ax in axs.flat[len(names):]:
        ax.axis('off')
    for ax, n in zip(axs.flat, names):
        t_, c = S.tracks[n], S.conds[n]
        col = TASK_COL[c.task]
        x = [tt.srs_db for tt in t_.trials]
        if x:
            ax.plot(range(1, len(x) + 1), x, '-', color=col, lw=1.1, alpha=.8)
            ax.scatter(range(1, len(x) + 1), x, s=13, zorder=3, edgecolors=col, lw=.8,
                       c=[col if tt.correct else 'white' for tt in t_.trials])
            rv = [(i + 1, tt.srs_db) for i, tt in enumerate(t_.trials) if tt.reversal]
            if rv:
                ax.plot(*zip(*rv), 'o', ms=8, mfc='none', mec=INK, mew=.6)
        thr = t_.threshold_db()
        if thr is not None:
            ax.axhline(thr, color=col, ls='--', lw=1)
        ax.axhline(CEIL['alone' if c.kind == 'alone' else c.task]['ceiling_srs_db'], color=RED, lw=.6, ls=':')
        ax.set_title(c.label(), fontsize=7.5, color=col)
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True, nbins=4))
        ax.tick_params(labelsize=6.5); ax.grid(alpha=.15)
    axs[0, 0].set_ylabel('SRS (dB)')
    fig.suptitle('your staircases  ·  filled = right, open = wrong, rings = reversals, dashed = threshold, '
                 'dotted red = shortcut ceiling', x=.01, ha='left', fontsize=8.5, color=MUTED, y=1.0)
    plt.tight_layout(); plt.show()

    # thresholds, binding index, weights
    thr = {n: S.tracks[n].threshold_db() for n in names}
    rows = S.log
    wts = AN.decision_weights(rows, cfgx)
    wc = AN.weight_curve(wts, cfgx)
    bc = AN.binding_curve(thr, cfgx)
    fig, axs = plt.subplots(1, 3, figsize=(11.6, 3.4))
    for task in 'AB':
        y = [thr.get(f'{task}_{s:g}'.replace('.', 'p')) for s in stepsx]
        y = [np.nan if v is None else v for v in y]
        axs[0].plot(stepsx, y, '-o', color=TASK_COL[task], lw=2, ms=6, label=f'type {task}')
        axs[0].axhline(CEIL[task]['ceiling_srs_db'], color=TASK_COL[task], ls=':', lw=1)
    for nm, mk in (('A_cap', 's'), ('B_cap', 's'), ('B_alone', 'D')):
        if thr.get(nm) is not None:
            axs[0].plot(stepsx[-1] + 10 + 6 * (nm == 'B_alone'), thr[nm], mk,
                        color=TASK_COL[nm[0]], ms=7, mfc='white' if nm == 'B_alone' else None)
    axs[0].set_ylabel('threshold, SRS (dB)'); axs[0].set_title('thresholds  (dotted: shortcut ceilings)')
    axs[0].legend(fontsize=7.5, loc='best')
    axs[1].plot(stepsx, bc['bi'], '-o', color=INK, lw=2, ms=6)
    if bc['capture']:
        axs[1].plot(0, bc['capture']['bi_capture'], 's', color=INK, mfc='white', ms=7, label='captors')
        axs[1].legend(fontsize=7.5)
    axs[1].axhline(0, color=MUTED, lw=.6)
    axs[1].set_ylabel('B minus A (dB)'); axs[1].set_title('binding index  (positive: bound)')
    if 'IMAGINARY' in globals():
        for name, (col, _) in WHO.items():
            _, w_im = IMAGINARY[name]
            axs[2].plot(STEPS, w_im['w'], '-', color=col, lw=4, alpha=.13)
    w = np.array(wc['w'], float)
    se = [wts.get(f'{s:g}', {}) for s in stepsx]
    axs[2].plot(stepsx, w, '-o', color=INK, lw=2.2, ms=6, label='you', zorder=4)
    for s, d in zip(stepsx, se):
        if d.get('ok'):
            bt, bo = d['b_timbre'], d['b_own']
            if bt + bo > 0:
                sw = math.sqrt((bo * d['se_timbre']) ** 2 + (bt * d['se_own']) ** 2) / (bt + bo) ** 2
                axs[2].plot([s, s], [d['w_timbre'] - 1.96 * sw, d['w_timbre'] + 1.96 * sw],
                            color=INK, lw=1, alpha=.5)
    if np.isfinite(wc['w_capture']):
        axs[2].plot(0, wc['w_capture'], 's', color=INK, mfc='white', ms=7, label='captors', zorder=4)
    if wc['s50_ms'] is not None:
        axs[2].axvline(wc['s50_ms'], color=RED, lw=1.2)
        axs[2].text(wc['s50_ms'] + 1, 1.08, f"your bound: {wc['s50_ms']:.0f} ms", color=RED, fontsize=8.5,
                    fontweight='bold')

    axs[2].set_ylim(-.15, 1.2); axs[2].set_ylabel('timbre weight')
    axs[2].set_title('decision weights  (1: one sound)'); axs[2].legend(fontsize=7.5, loc='lower left')
    for ax in axs:
        ax.set_xticks(stepsx); ax.set_xlabel('shear (ms)'); ax.grid(alpha=.2)
    extra = [(stepsx[-1] + 10, 'cap')] * any(thr.get(n) is not None for n in ('A_cap', 'B_cap')) \
        + [(stepsx[-1] + 16, 'alone')] * (thr.get('B_alone') is not None)
    if extra:
        axs[0].set_xticks(list(stepsx) + [x for x, _ in extra])
        axs[0].set_xticklabels([f'{s:g}' for s in stepsx] + [l for _, l in extra])
    plt.tight_layout(); plt.show()

    catch = [r for r in rows if r['phase'] == 'catch']
    main = [r for r in rows if r['phase'] == 'main']
    lines = [f'{len(main)} trials on the staircases, {len(catch)} catch trials '
             + (f'({100 * np.mean([r["correct"] for r in catch]):.0f}% right; the lab needs at least 85%)'
                if catch else ''),
             f'{sum(t_.finished for t_ in S.tracks.values())} of {len(names)} staircases finished']
    if wc['s50_ms'] is not None and len(stepsx) < 5:
        lines.append(f'with {len(stepsx)} steps the bound is interpolated over wide gaps, so treat '
                     f"{wc['s50_ms']:.0f} ms as rough")
    if wc['s50_ms'] is None:
        lines.append('your bound is not defined yet: the timbre weight has to fall by at least '
                     f'{AN.MIN_SWING_W:g} from the smallest to the largest shear')
    f_ = AN.scaled_model_fit(stepsx, wc['w'], cfgx, 'halfwave')
    if f_.get('ok') and len(stepsx) >= 3:
        lines.append(f"stretching the model's curve to fit you: alpha {f_['alpha']:.2f} "
                     '(1 = binding lasts exactly as long as the half-wave model says)')
    display(HTML('<div style="font:13px system-ui;color:#1f2933;line-height:1.6">'
                 + '<br>'.join(lines) + '</div>'))

    out = Path('tcbound_my_session.csv')
    with out.open('w', newline='') as f:
        wr = csv.DictWriter(f, fieldnames=list(rows[0])); wr.writeheader(); wr.writerows(rows)
    b_dl = W.Button(description=f'download {out.name}', icon='download')
    def _dl(_):
        try:
            from google.colab import files
            files.download(str(out))
        except Exception:
            print(f'saved to {out.resolve()}')
    b_dl.on_click(_dl)
    display(b_dl)

---
# 10 · Build your own

Change anything and run. Every number here goes through the same code as the experiment, so if a
setting would break the design (tones that no longer fit in a repetition, a level that clips) you
are told why.

In [ ]:
#@title build a trial
trial_type   = 'A'    #@param ["A", "B", "none"]
captors      = False  #@param {type:"boolean"}
red_alone    = False  #@param {type:"boolean"}
shear_ms     = 24     #@param {type:"number"}
louder_db    = 6      #@param {type:"number"}
rove_db      = 12     #@param {type:"number"}
tone_ms      = 60     #@param {type:"number"}
rate_hz      = 3.0    #@param {type:"number"}
repetitions  = 5      #@param {type:"integer"}

try:
    cfg_b = CFG.replace(rove_db=float(rove_db), tone_ms=float(tone_ms), rate_hz=float(rate_hz),
                        n_repeats=int(repetitions), steps_ms=(0.0, float(shear_ms)))
    validate(cfg_b)
    kind = 'alone' if red_alone else (f'{trial_type}_cap' if captors and trial_type != 'none' else trial_type)
    c_b = cond(kind, shear_ms)
    tr_b = (build_trial(cfg_b, c_b, srs_db(min(louder_db, cfg_b.delta_max_db)), np.random.default_rng(),
                        target_position=1) if louder_db > 0 else
            build_trial(cfg_b, c_b, cfg_b.srs_min_db, np.random.default_rng(), target_position=1,
                        delta_override_db=0.0))
    R_b = None
    if c_b.kind != 'alone':
        cc = Condition('m', 'A', float(shear_ms), c_b.kind)
        M = np.asarray(coherence_matrix(envelopes(cfg_b, cc), MODEL_FS).matrix, float)
        dd = np.sqrt(np.clip(np.diag(M), 1e-300, None)); R_b = M / np.outer(dd, dd)
    pair_figure(tr_b, cfg_b, web_R=R_b)
    pairs([('', TASK_COL.get(trial_type, INK), *sounds(tr_b, cfg_b))])
    if trial_type in 'AB' and not red_alone:
        ce = ceilings(cfg_b, c_b, n=6000)
        print(f"the easiest shortcut ({CUE_LABELS[ce['lowest_shortcut']]}) needs {ce['ceiling_db']:.1f} dB "
              f"for 79% at a rove of ±{rove_db:g} dB")
except (ConfigError, ValueError) as e:
    print('not possible:', e)

---
# 11 · For modellers

A batch of trials for any condition, as arrays: both sounds, the level of every tone, the answer,
and the tone envelopes at 1 kHz (the input the coherence model uses). For a real session,
`python -m tcbound export` rebuilds exactly the trials a listener heard, response by response.

In [ ]:
#@title make a batch
condition   = 'B_16'   #@param ["A_0", "B_0", "A_8", "B_8", "A_16", "B_16", "A_32", "B_32", "A_64", "B_64", "A_cap", "B_cap", "B_alone"]
louder_db   = 4        #@param {type:"number"}
n_trials    = 64       #@param {type:"integer"}
seed        = 0        #@param {type:"integer"}

c_m = {c.name: c for c in conditions(CFG)}[condition]
rng_m = np.random.default_rng(seed)
batch = [build_trial(CFG, c_m, srs_db(louder_db), rng_m) for _ in range(int(n_trials))]
L_ = max(len(trial_wave(b_)) for b_ in batch)

def _env(cfg, iv, fs=MODEL_FS):
    n = int(round(iv.onsets_ms.shape[0] * cfg.period_ms + 400))
    e = np.zeros((cfg.n_tones, n)); w = int(round(cfg.tone_ms * fs / 1000)); r = int(round(cfg.ramp_ms * fs / 1000))
    shape = np.ones(w); shape[:r] = np.linspace(0, 1, r); shape[-r:] = np.linspace(1, 0, r)
    tones = [(iv.onsets_ms, iv.gains_db)]
    if iv.captor_onsets_ms is not None:
        tones.append((iv.captor_onsets_ms, iv.captor_gains_db))
    for on, g in tones:
        for rep in range(on.shape[0]):
            for k in range(cfg.n_tones):
                if np.isfinite(on[rep, k]):
                    i = int(round(on[rep, k] * fs / 1000)); e[k, i:i + w] += shape * 10 ** (g[rep, k] / 20)
    return e

waves = np.stack([np.pad(trial_wave(b_), (0, L_ - len(trial_wave(b_)))) for b_ in batch]).astype('float32')
gains = np.stack([[b_.first.gains_db, b_.second.gains_db] for b_ in batch])
envs = np.stack([[_env(CFG, b_.first), _env(CFG, b_.second)] for b_ in batch]).astype('float32')
answer = np.array([b_.target_position for b_ in batch])
out_m = Path(f'tcbound_batch_{condition}_{louder_db:g}dB.npz')
np.savez_compressed(out_m, waves=waves, gains_db=gains, envelopes=envs, answer=answer,
                    sample_rate=SR, envelope_rate=MODEL_FS, freqs_hz=np.array(FREQS),
                    condition=condition, louder_db=louder_db)
print(f'waves      {waves.shape}   both sounds and the pause, {SR} Hz')
print(f'gains_db   {gains.shape}   trial, sound, repetition, tone')
print(f'envelopes  {envs.shape}   trial, sound, tone, ms')
print(f'answer     {answer.shape}   which sound had the louder red tone')
print(f'saved {out_m.resolve()}  ({out_m.stat().st_size / 1e6:.1f} MB)')
def _dl_m(_):
    try:
        from google.colab import files
        files.download(str(out_m))
    except Exception:
        print(f'saved to {out_m.resolve()}')
b_m = W.Button(description=f'download {out_m.name}', icon='download')
b_m.on_click(_dl_m)
display(b_m)